In [0]:
# ATTRIBUTES WITH AT LEAST 1 NON-VALUE

# import needed packages and functions
from pyspark.sql import functions as F

# import tables
## silver customer table
customer = spark.table("crm_reporting.dim_customer")

## customer bridge table
customer = spark.table("crm_reporting.dim_customer")

## bmdm table
gdm = spark.table("crm_reporting.dim_gdm_brand_profile")

# df = gdm.filter((col('registration_sub_source') == 'product finder') & \
#                        (col('brand_code') == 'LRP') & \
#                        (col('brand_country') == 'BRA'))

In [ ]:
## DIFERENCIAR ENTRE VARIABLES
a= ' Lower part of the face'
b= 'Lower part of the face'

# Convertir a bytes y luego a representación hexadecimal
a_hex = a.encode("utf-8").hex()
b_hex = b.encode("utf-8").hex()

print('iguales?: ',a_hex == b_hex)
print('a=',a_hex,'b=',b_hex)

: 

In [0]:

df = customer.withColumn('source_name', F.upper('source_name')).\
              withColumn('registration_sub_source', F.upper('registration_sub_source')).\
              filter(F.col('source_name').isin('DEMANDWARE','JEBBIT')).\
              select('registration_sub_source').distinct().\
              orderBy('registration_sub_source')                  

display(df)



#display(df)

# # Contar valores no nulos para cada columna
# agg_expr = [count(when(col(c).isNotNull(), c)).alias(c) for c in df.columns]

# # Ejecutar la agregación
# non_null_counts = df.agg(*agg_expr).collect()[0]

# # Filtrar columnas que no son completamente nulas
# non_completely_null_columns = [c for c in df.columns if non_null_counts[c] > 0]

# # Mostrar las columnas que no están completamente vacías
# print(len(non_completely_null_columns))
# print(non_completely_null_columns)
# 'cart checkout'
# 'сart checkout'

registration_sub_source
null
""
CART CHECKOUT
CONTACT US
CONTEST
FOOTER
HAIR QUIZ
HEADER
POPUP
PRODUCT FINDER


In [0]:
# Es probable que las dos entradas similares, "cart checkout" y "сart checkout", aparezcan como distintas debido a diferencias sutiles en los caracteres, aunque a simple vista se vean iguales.
#  La diferencia más común en este tipo de casos es que uno de los caracteres pertenece a un alfabeto diferente, como el cirílico en lugar del latino.

# Por ejemplo:

# "c" en "cart checkout" es del alfabeto latino.
# "с" en "сart checkout" podría ser una letra del alfabeto cirílico (parece la letra latina "c", pero es un carácter distinto).
# Puedes comprobar esto utilizando alguna función que compare los valores a nivel de código ASCII o Unicode, para identificar las diferencias. Aquí te muestro cómo podrías hacerlo en PySpark:
  
df.select("registration_sub_source", F.hex(F.col("registration_sub_source").cast("binary")).alias("unicode")).distinct().show(truncate=False)
#'41525420434845434B4F5554'
#'41525420434845434B4F5554'

## comparar

+------------------------+------------------------------------------------+
|registration_sub_source |unicode                                         |
+------------------------+------------------------------------------------+
|NULL                    |NULL                                            |
|                        |                                                |
|CART CHECKOUT           |4341525420434845434B4F5554                      |
|CONTACT US              |434F4E54414354205553                            |
|CONTEST                 |434F4E54455354                                  |
|FOOTER                  |464F4F544552                                    |
|HAIR QUIZ               |48414952205155495A                              |
|HEADER                  |484541444552                                    |
|POPUP                   |504F505550                                      |
|PRODUCT FINDER          |50524F445543542046494E444552                    |
|QUIZZES/DIA

In [0]:
# Convertir todos los valores a minúsculas y eliminar caracteres no deseados
df = df.withColumn("registration_sub_source", F.regexp_replace(F.col("registration_sub_source"), "[^\\x00-\\x7F]", "")).\
        withColumn("registration_sub_source", F.trim(F.col("registration_sub_source"))).\
        withColumn("registration_sub_source", F.regexp_replace(F.col("registration_sub_source"), "[\\u200B\\u00A0]+", ""))

# Luego hacer un distinct nuevamente
df_distinct = df.select("registration_sub_source").distinct()

display(df_distinct)

registration_sub_source
CART CHECKOUT
REGISTRATION
null
POPUP
SKIN DR
ART CHECKOUT
CONTACT US
PRODUCT FINDER
HEADER
FOOTER


In [0]:
df.filter(F.length(F.col("registration_sub_source")) > 0).select(
    "registration_sub_source",
    F.ascii(F.col("registration_sub_source")[0]).alias("ascii_value"),
    F.hex(F.col("registration_sub_source").cast("binary")).alias("unicode_value")
).distinct().show(truncate=False)

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-1066504240448149>, line 1
----> 1 df.filter(F.length(F.col("registration_sub_source")) > 0).select(
      2     "registration_sub_source",
      3     F.ascii(F.col("registration_sub_source")[0]).alias("ascii_value"),
      4     F.hex(F.col("registration_sub_source").cast("binary")).alias("unicode_value")
      5 ).distinct().show(truncate=False)

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:1161, in DataFrame.show(self, n, truncate, vertical)
   1160 def show(self, n: int = 20, truncate: Union[bool, int] = True, vertical: bool = False) -> None:
-> 1161     print(self._show_string(n, truncate, vertical))

File /databricks/spark/python/pyspark/sql/connect/dataframe.py:905, in DataFrame._show_string(self, n, truncate, vertical)
    888     except ValueError:
    889         raise PySparkTypeError(
    890 